In [1]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

In [2]:
def retrieve_documents(dataset_id, api_key, query, search_method="semantic_search", weights=0.5,top_k=3):
    """
    从数据集中检索文档。

    参数:
    - dataset_id (str): 数据集 ID。
    - api_key (str): API 密钥。
    - query (str): 查询关键词。
    - search_method (str): 检索方法，默认为 "keyword_search"。
    - top_k (int): 返回的结果数量，默认为 1。

    返回:
    - dict: API 的响应结果。
    """
    # 设置请求 URL 和头部
    url = f"http://172.16.2.77:56966/v1/datasets/{dataset_id}/retrieve"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 定义请求体
    payload = {
        "query": query,
        "retrieval_model": {
            "search_method": search_method,
            "reranking_enable": False,
            "reranking_mode": None,
            "reranking_model": {
                "reranking_provider_name": "",
                "reranking_model_name": ""
            },
            "weights": weights,
            "top_k": top_k,
            "score_threshold_enabled": False,
            "score_threshold": None
        }
    }

    # 发送 POST 请求
    response = requests.post(url, headers=headers, data=json.dumps(payload))

    # 返回解析后的响应
    if response.status_code == 200:
        return response.json()
    else:
        print({"error": response.status_code, "message": response.text})
        return {"error": response.status_code, "message": response.text}


In [24]:
# 使用示例
# dataset_id = "88a91412-da40-4892-b274-d9afe4880d2f"  #核保知识
# dataset_id = "88e7f501-0511-45e6-831c-c966360cea2d"  # 昆仑健康核保
dataset_id = "c02c5910-993a-4bed-8985-c7f23572b837"
api_key = "dataset-F8zSYohCngBjVNsh2NGBv5Bl"  # 替换为实际 API 密钥
query = "甲状腺炎"

result = retrieve_documents(dataset_id, api_key, query)
print(result)
print("实际返回结果量：",len(result["records"]))

for rel in result["records"]:
    print(rel["segment"]["document"])
    print(rel["segment"]["content"])
    print(len(rel["segment"]["content"]))

    print(rel["score"])

{'query': {'content': '甲状腺炎'}, 'records': [{'segment': {'id': 'ad468124-669b-4c9f-ba8c-fe7318b77e1c', 'position': 2, 'document_id': 'be5c2e8b-29e8-4fc2-903c-f54e80392e2a', 'content': "{'疾病': '甲状腺炎', '': '亚急性甲状腺炎', '资料': '病历、甲状腺超声、甲状腺功能', '检查结果': '病情控制稳定6个月，甲状腺功能及超声正常', '重疾': '标体', '防癌': '标体', '意外险': '标体', '护理险': '标体', '医疗险': '除外'}", 'answer': None, 'word_count': 148, 'tokens': 218, 'keywords': ['意外险', '甲状腺', '重疾', '防癌', '亚急性', '标体', '医疗险', '超声', '甲状腺炎', '功能'], 'index_node_id': '294c4aab-606f-4f17-9c32-b5a0cefee707', 'index_node_hash': '0ea86242b1fd79d8672bedcd65baf984056754d61bb3e9979809b61e26dd12f7', 'hit_count': 0, 'enabled': True, 'disabled_at': None, 'disabled_by': None, 'status': 'completed', 'created_by': '71416823-659b-4a69-a747-9b2fe3670645', 'created_at': 1732243896, 'indexing_at': 1732243895, 'completed_at': 1732243896, 'error': None, 'stopped_at': None, 'document': {'id': 'be5c2e8b-29e8-4fc2-903c-f54e80392e2a', 'data_source_type': 'upload_file', 'name': '甲状腺疾病.txt', 'doc_typ

# 大模型支持

In [21]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-7B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

In [5]:
user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input}]
qa_base(input)

'中国的首都是北京。'

# 核保知识重建

In [6]:
# colum:
colums_v1 = ['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '防癌险', '护理险', '医疗险', '意外险', '备注']
colums_v2 = ["疾病","资料","检查结果"]
save_data_path = "./save_data"
files_v1 = []
files_v2 = []
files_v3 = []
files_other = []
for file in os.listdir(save_data_path):
    print(file)
    file_path = os.path.join(save_data_path,file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    # print(datas)
    one_data = datas[0].strip()
    dict_data = eval(one_data)
    for key in dict_data.keys():
        print(key)
        if key in colums_v1[:1]:
            files_v1.append(file)
        elif key in colums_v2[:1]:
            files_v2.append(file)
        elif key == "诊断":
            files_v3.append(file)
        else:
            files_other.append(file)
        break

肝脏.txt
疾病
呼吸.txt
疾病名称
乳腺.txt
疾病
五官科.txt
疾病名称
风湿免疫内分泌.txt
疾病名称
肺结节.txt
诊断
甲状腺疾病.txt
疾病
新生儿.txt
疾病名称
外科.txt
疾病名称
心血管疾病.txt
诊断
消化系统其他.txt
疾病名称
血液系统.txt
疾病名称
上消.txt
疾病
感染.txt
疾病名称
血压高.txt
诊断
胆囊.txt
疾病名称
宫颈.txt
器官
子宫.txt
疾病
妇科肿物.txt
疾病
甲状腺癌.txt
诊断
神经系统.txt
疾病名称
泌尿.txt
疾病名称
肠道.txt
诊断
阴道.txt
器官


In [7]:
print(len(files_v1))
print(colums_v1)
print(f"files_v1:{files_v1}")
print(len(files_v2))
print(colums_v2)
print(f"files_v2:{files_v2}")
print(len(files_v3))
print("诊断")
print(f"files_v3:{files_v3}")
print(len(files_other))
print(f"file other:{files_other}")
for file in files_v1:
    file_path = os.path.join(save_data_path,file)
    print(file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    # print(datas)
    one_data = datas[0].strip()
    dict_data = eval(one_data)
    print(dict_data.keys())

print("\n\n")

for file in files_v2:
    file_path = os.path.join(save_data_path,file)
    print(file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    # print(datas)
    one_data = datas[0].strip()
    dict_data = eval(one_data)
    print(dict_data.keys())

print("\n\n")
for file in files_v3:
    file_path = os.path.join(save_data_path,file)
    print(file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    # print(datas)
    one_data = datas[0].strip()
    dict_data = eval(one_data)
    print(dict_data.keys())

print("\n\n")

for file in files_other:
    file_path = os.path.join(save_data_path,file)
    print(file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    # print(datas)
    one_data = datas[0].strip()
    dict_data = eval(one_data)
    print(dict_data.keys())

11
['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '防癌险', '护理险', '医疗险', '意外险', '备注']
files_v1:['呼吸.txt', '五官科.txt', '风湿免疫内分泌.txt', '新生儿.txt', '外科.txt', '消化系统其他.txt', '血液系统.txt', '感染.txt', '胆囊.txt', '神经系统.txt', '泌尿.txt']
6
['疾病', '资料', '检查结果']
files_v2:['肝脏.txt', '乳腺.txt', '甲状腺疾病.txt', '上消.txt', '子宫.txt', '妇科肿物.txt']
5
诊断
files_v3:['肺结节.txt', '心血管疾病.txt', '血压高.txt', '甲状腺癌.txt', '肠道.txt']
2
file other:['宫颈.txt', '阴道.txt']
呼吸.txt
dict_keys(['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '防癌险', '护理险', '医疗险', '意外险', '备注'])
五官科.txt
dict_keys(['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '意外险', '护理', '防癌险', '医疗险', '备注'])
风湿免疫内分泌.txt
dict_keys(['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '护理险', '医疗险', '意外险', '防癌险', '备注'])
新生儿.txt
dict_keys(['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '护理', '医疗险', '意外险', '防癌险', '备注'])
外科.txt
dict_keys(['疾病名称', '诊断', '病情现状', '', '体检项目', '项目结果', '重疾险', '护理', '防癌险', '意外险', '医疗险', '备注'])
消化系统其他.txt
dict_keys(['疾病名称', '诊断', '病情现状', '体检项目', '项目结果', '重疾险', '防癌', '护理险', 

In [8]:
all_data = []
print(f"files v1:{len(files_v1)}")
for file in files_v1:
    file_path = os.path.join(save_data_path,file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    for one_data in datas:
        save_dict = {}
        one_data = one_data.strip()
        if one_data == "":
            continue
        dict_data = eval(one_data)
        disease_name = dict_data["疾病名称"]
        if disease_name == "":
            continue
        save_dict["disease_name"] = disease_name
        save_dict["file_name"] = file
        save_dict["conclusion"] = one_data
        all_data.append(save_dict)

print(f"all_data:{len(all_data)}")

print(f"files v2:{len(files_v2)}")
for file in files_v2:
    file_path = os.path.join(save_data_path,file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    for one_data in datas:
        save_dict = {}
        one_data = one_data.strip()
        if one_data == "":
            continue
        dict_data = eval(one_data)
        disease_name = dict_data["疾病"]
        if disease_name == "":
            continue
        save_dict["disease_name"] = disease_name
        save_dict["file_name"] = file
        save_dict["conclusion"] = one_data
        all_data.append(save_dict)

print(len(all_data))


print(f"files v3:{len(files_v3)}")
for file in files_v3:
    file_path = os.path.join(save_data_path,file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    for one_data in datas:
        save_dict = {}
        one_data = one_data.strip()
        if one_data == "":
            continue
        dict_data = eval(one_data)
        disease_name = dict_data["诊断"]
        if disease_name == "":
            continue
        save_dict["disease_name"] = disease_name
        save_dict["file_name"] = file
        save_dict["conclusion"] = one_data
        all_data.append(save_dict)


print(len(all_data))

print(f"files other:{len(files_other)}")
for file in files_other:
    file_path = os.path.join(save_data_path,file)
    with open(file_path,"r")as f:
        datas = f.readlines()
    for one_data in datas:
        save_dict = {}
        one_data = one_data.strip()
        if one_data == "":
            continue
        dict_data = eval(one_data)
        disease_name = dict_data["疾病"]
        if disease_name == "":
            continue
        save_dict["disease_name"] = disease_name
        save_dict["file_name"] = file
        save_dict["conclusion"] = one_data
        all_data.append(save_dict)


print(len(all_data))

files v1:11
all_data:719
files v2:6
867
files v3:5
1032
files other:2
1061


In [9]:
df_data_all = pd.DataFrame(all_data)

In [10]:
print(df_data_all.shape)
df_data_all.head()

(1061, 3)


,disease_name,file_name,conclusion
0,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
1,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
2,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
3,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."
4,肺大疱（肺囊肿）,呼吸.txt,"{'疾病名称': '肺大疱（肺囊肿）', '诊断': '未接受手术，连续 X 射线检查显示病..."


In [47]:
# df_data_all.to_csv("disease_2_conclusion.csv",index=False)

In [11]:
all_disease_name = set(df_data_all["disease_name"].tolist())
print(len(all_disease_name))
print(all_disease_name)

235
{'病理分类： 乳头状癌、滤泡性癌（55岁以下）', '主动脉瓣疾病', '霉菌性阴道炎', '中耳炎', '过敏性紫癜', '房间隔缺损', '短暂性脑缺血发作', '甲减', '肠吸收不良', '腰椎滑脱', '扩张性心肌病', '风湿性关节炎', '多发性硬化', '异位妊娠', '腰椎间盘突出', '腮腺良性肿瘤', '胆囊炎', '无明确诊断，与既往影像对比增大或有恶性倾向', '肠梗阻', '乳腺结节、囊肿、占位、异常回声', '心肌病家族史', '（下肢）静脉曲张', '前列腺炎', '消化性溃疡', '腺瘤、息肉', '颅咽管瘤', '肛瘘', '新生儿心肌损害', '新生儿呼吸窘迫综合征', '退行性改变', '青光眼', '慢性肾小球肾炎', '臂丛神经痛', '急性失血性贫血', '胸膜间皮瘤', '酒精性肝病', '腹膜炎', '慢性肾盂肾炎', '肝血管瘤', '已接受手术，有明确诊断', '心肌炎', '面神经炎\n（面瘫）', '脾大', '胰腺炎', '胃炎', '视网膜中央静脉血栓形成', '巨结肠', '泌尿系结石（无高血压和肾功能损害）', '肛周脓肿', '支气管扩张', '视网膜病的病因未明，正在进行检查', '肾病综合征（慢性）', '甲状腺结节', '子宫脱垂', '肝硬化', '慢性结/直肠炎/溃疡性结肠炎', '肌营养不良症', '食管', '病理分类：间变性癌、未分化癌、髓样癌等', '病理分类： 乳头状癌、滤泡性癌（55岁以上）', '气管炎/支气管炎', '扁桃体炎', '脾功能亢进', '系统性红斑狼疮', '纵膈肿瘤', 'EB病毒感染', '哮喘', '股骨头坏死', '美尼尔氏病', '耳鸣', '急性细菌性肾盂肾炎、急性肾盂肾炎', '三叉神经痛', '盆腔积液', '梅毒', '抑郁症', '心脏肥大/增大/扩大', '肝内胆管结石', '烧伤', '颈椎病', '甲状腺炎', '胸膜炎', '胆囊息肉', '新生儿卵圆孔未闭', '呼吸衰竭', '肺错构瘤', '胆脂瘤', '缺血性脑血管病', '无明确诊断，见于当前影像，与既往2-3次影像对比无改变或变小，无钱币状、磨玻璃状、分叶状及其他恶性倾向', '甲亢', '结直肠癌', '肺炎\n（不包括新冠肺炎）', '其他心脏结构异常', 

In [12]:
df_disease_map = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)

In [13]:
disease_n = set(df_disease_map["disease_n"].tolist())
print(len(disease_n))
disease_intersection = disease_n & all_disease_name
print(len(disease_intersection))

769
111


In [14]:
df_disease_map.head()

,def_name,HBZS_def,HBZS_did,disease_n,disease_list,kwords,CI,MI,ADB,Life,ZZ_match_disease,RZ_match_disease,MZ_match_disease,KL_match_disease
0,JiZhenL,,,肌阵挛,['肌阵挛'],,,,,,dianxian,,,
1,JiaZX_MiMXBB,JiaZXMMXBB,135,甲状腺弥漫性病变,['甲状腺弥漫性病变'],"[['甲状腺', '甲状', '甲壮腺', '甲超'], ['弥漫性', '非均质性改变',...",甲状腺恶性肿瘤（含原位癌）及其复发和转移,甲状腺疾病,,甲状腺恶性肿瘤（含原位癌）及其复发和转移,jiazhuangxianzhongliu,liangxingjiazhuangxianjibing,,
2,JiaZX_QieCSH,,,甲状腺切除术后,['甲状腺切除术后'],,,,,,,liangxingjiazhuangxianjibing,,
3,ZhuiTWX_ZHZ,,,椎体外系综合征,['椎体外系综合征'],,,,,,,,,
4,NongDX_NaoB,,,脓毒症相关性脑病,['脓毒症相关性脑病'],,,ruxian,,,,,,


# 对测试用例进行RAG生成

In [18]:
df = pd.read_csv("./规则引擎对比结果-评点数据.csv",keep_default_na=False)

In [17]:
df["临床诊断（化验项、疾病等以下划线拼接）"].tolist()

['宫腔内稍高回声团,考虑子宫内膜息肉',
 '子宫肌瘤可能',
 '双肾结石(沙粒样)',
 '脂肪肝（中度）',
 '双肾结石',
 '右侧甲状腺囊性结节',
 '胆囊息肉',
 '右侧甲状腺下极下方囊性灶，  考虑甲状旁腺来源可能',
 '肝血管瘤',
 '肝囊肿',
 '双肺上叶微小结节，多为增殖灶',
 '双肾结晶',
 '脂肪肝',
 '右侧乳腺结节待查',
 '右侧乳腺结节待查',
 '双侧甲状腺结节',
 '多发性胆囊息肉',
 '左肺上叶前段磨玻璃影',
 '甲状腺右叶囊肿(ACRTI-RADS1类)',
 '甲状腺左叶结节(ACRTI-RADS3类)',
 '肝内胆管结石(多发)',
 '两肺多发小结节',
 '右乳囊性结节拟US-BI-RADS2类',
 '甲状腺右叶结节',
 '右肺微小结节',
 '轻度脂肪肝',
 '双肾结晶',
 '窦性心律不齐',
 '桥本氏甲状腺炎',
 '颈椎病',
 '双肾结晶',
 '肺动脉少量反流']

In [16]:
for input_dise in df["临床诊断（化验项、疾病等以下划线拼接）"].tolist():
    if input_dise in all_disease_name:
        print(input_dise)

胆囊息肉
肝血管瘤
肝囊肿
颈椎病


In [60]:
print(df.shape)
df.head()

(32, 15)


,姓名,编号（身份证号）,性别\n（1:男，2:女，0:未知）,医院,日期,临床诊断（化验项、疾病等以下划线拼接）,账单金额（数字）,年龄,编号（案件编号）,图片名/文件名,attach id,图片分类/票据类别,影像报告内容image_report,data_source：类别 如昆仑体检告知,线上页面与结果
0,张颖,421023199012060444,2,市到大学深圳医院,,"宫腔内稍高回声团,考虑子宫内膜息肉",,34,M202444111993763,ile_TWzoqlM3_20241119142423324.jpg,,,"{'report_name': '', 'des_dic': '经阴道三维超声检查：\\n后...",体检,URL http://yc-dev.smart-insight-service.com:41...
1,马春兰,330621199503154709,2,,,子宫肌瘤可能,,29,M202431111993911,file_4BqWBucI_20241119194907173.jpg,,,"{'report_name': '', 'des_dic': '【子宫】 【经阴道】\\n...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
2,高刚正,440303198910301317,1,,,双肾结石(沙粒样),,35,M202444111893402,file_Oj01H8zK_20241118225440966.jpg,,,"{'report_name': '', 'des_dic': '双肾轮廓清晰。 切面形态大...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
3,高刚正,440303198910301317,1,,,脂肪肝（中度）,,35,M202444111893402,file_Oj01H8zK_20241118225440966.jpg,,,"{'report_name': '', 'des_dic': '肝脏切面轮廊清晰， 右叶斜...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...
4,郑序华,440582199008156187,2,,,双肾结石,,34,M202444111893285,file_n3wYhN9Y_20241118142225174.jpg,,,"{'report_name': '', 'des_dic': '双肾形态大小位置正常 包膜...",体检,URL\nhttp://yc-dev.smart-insight-service.com:4...


In [19]:
df.columns

Index(['姓名', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果'],
      dtype='object')

In [56]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_time_use = []
for idx,row in df.iterrows():
    
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    print("病人基本信息",basic_info)
    print(image_report)
    if image_report != "":
        image_report = json.loads(image_report)
        print(type(image_report))
        query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    else:
        query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    # query = diagnose
    
    s_time = time.time()
    try:
        recall_kb = retrieve_documents(dataset_id, api_key, query,search_method="semantic_search",top_k=5)
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)

        # print("核保结论实际召回量：",len(recall_kb["records"]))
        ans_list = [rel["segment"]["content"] for rel in recall_kb["records"]]
        # print("召回的核保结论：",ans_list)
    except Exception as es:
        print(es)
        ans_list = []

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,output_struct=output_struct)
    # print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)

    results_underwriting.append(result)
    recall_query.append(ans_list)

    

病人基本信息 年龄:34,性别:女性,临床诊断:宫腔内稍高回声团,考虑子宫内膜息肉,影像报告:{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
<class 'dict'>
recall time use: 0.22617173194885254
病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低

In [52]:
sum(recall_time_use)/len(recall_time_use)

0.47619374841451645

# 对RAG结果进行check

In [53]:
print(len(results_underwriting))
print(len(recall_query))

32
32


In [57]:
df["RAG核保结论"] = results_underwriting
df["recall_query"] = recall_query

In [58]:
df.to_csv("核保结论_规则引擎_vs_RAG_SemanticSearch.csv",index=False)

In [81]:
def remove_urls(text):
    # 正则表达式匹配URL
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    # 替换掉所有匹配的URL
    return re.sub(url_pattern, '', text)


for index,row in df.iterrows():
    print(index)
    online_ans = row["线上页面与结果"]
    rag_ans = row["RAG核保结论"]
    print("on line ans:",remove_urls(online_ans))
    print("rag ans:",rag_ans)

0
on line ans: URL 
重疾 延期至术后或痊愈后
医疗 延期至术后或痊愈后
rag ans: {'重疾险': '延期至术后或痊愈后', '防癌': '延期至术后或痊愈后', '护理险': '延期至术后或痊愈后', '医疗险': '延期至术后或痊愈后', '意外险': '延期至术后或痊愈后'}
1
on line ans: URL

1
重疾 +0 
医疗 对子宫疾病及其并发症除外
rag ans: {'重疾险': '标体', '防癌': '标体', '护理险': '标体', '医疗险': '除外*1', '意外险': '标体'}
2
on line ans: URL

重疾 +0
医疗 除外泌尿系结石及其并发症
rag ans: {'重疾险': '延期', '防癌': '标体', '护理': '标体', '医疗险': '延期', '意外险': '标体'}
3
on line ans: URL

重疾 延期
医疗 延期
rag ans: {'重疾险': '标体', '防癌': '标体', '护理险': '标体', '医疗险': '除外肝脏疾病', '意外险': '标体'}
4
on line ans: URL

重疾 +0
医疗 除外泌尿系结石及其并发症
rag ans: {'重疾险': '延期', '防癌': '标体', '护理': '标体', '医疗险': '延期', '意外险': '标体'}
5
on line ans: URL

重疾 +0
医疗 除外
rag ans: {'重疾险': '延期', '防癌': '延期', '护理险': '延期', '医疗险': '延期', '意外险': '延期'}
6
on line ans: URL

重疾 补充资料：B超
医疗 补充资料：B超
rag ans: {'重疾险': '延期', '防癌险': '延期', '护理险': '延期', '医疗险': '延期', '意外险': '标体'}
7
on line ans: URL

重疾 延期
医疗 延期
rag ans: {'重疾险': '延期', '防癌': '延期', '护理险': '延期', '医疗险': '延期', '意外险': '延期'}
8
on line ans: URL

重疾 拒保
医疗 拒保
rag ans: {'重疾险': '标体', 

# Dify大模型本地部署
- ChatGLM,可以部署LLM
- OpenLLM,可以部署LLM,embedding
